# Lưu adapter B6 (không GPU, không Internet)

In [ ]:
import hashlib, json, shutil
from pathlib import Path

ds = sorted(Path("/kaggle/input").rglob("b6_trang_thai.json"))
assert len(ds) == 1, ds
b6 = ds[0].parent
st = json.loads(ds[0].read_text(encoding="utf-8"))
assert st.get("xong") is True, "B6 chua xong"
OUT = Path("/kaggle/working/b6_luu")
OUT.mkdir(parents=True)
shutil.copy2(ds[0], OUT / ds[0].name)
for ten, v in st["cau_hinh"].items():
    for e in range(1, v["epoch_xong"] + 1):
        vao = b6 / ten / f"ep{e}"
        assert (vao / "adapter_model.safetensors").exists(), vao
        ra = OUT / ten / f"ep{e}"
        ra.mkdir(parents=True)
        for f in ("adapter_model.safetensors", "adapter_config.json"):
            shutil.copy2(vao / f, ra / f)
    for f in sorted((b6 / ten).glob("sinh_ep*.jsonl")):
        shutil.copy2(f, OUT / ten / f.name)
mf = {}
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        h = hashlib.sha256()
        with open(p, "rb") as a:
            for k in iter(lambda: a.read(2**20), b""):
                h.update(k)
        mf[str(p.relative_to(OUT)).replace("\\", "/")] = {"byte": p.stat().st_size, "sha256": h.hexdigest()}
(OUT / "MANIFEST.json").write_text(json.dumps(mf, indent=1), encoding="utf-8")
print(len(mf), "file", round(sum(v["byte"] for v in mf.values()) / 2**30, 2), "GiB", flush=True)
print("XONG", flush=True)
